## **Fine Tuning**

In [ ]:
from transformers import AutoModelForCausalLM,AutoTokenizer

In [ ]:
model=AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")
tokenizer=AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from datasets import load_dataset
# Loading data.json instead of processing PDF
dataset = load_dataset('json', data_files='/content/drive/MyDrive/Colab Notebooks/finetune/data.json')

In [ ]:
print(dataset)

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

def preprocess_function(examples):
    # Concatenate 'input' and 'output' to form the full sequence for training
    # For causal language modeling, we typically train on the entire sequence,
    # and the labels are the input_ids themselves (shifted internally by the model)
    full_text = [inp + tokenizer.eos_token + out for inp, out in zip(examples["input"], examples["output"])]
    tokenized_inputs = tokenizer(
        full_text,
        truncation=True,
        padding="max_length",
        max_length=tokenizer.model_max_length if tokenizer.model_max_length > 0 else 512 # Set a reasonable max_length
    )
    # For causal language models, the labels are typically the input_ids
    tokenized_inputs["labels"] = tokenized_inputs["input_ids"].copy()
    return tokenized_inputs

# Apply the preprocessing function to the dataset
tokenized_dataset = dataset.map(preprocess_function, batched=True)

# The Trainer expects a Dataset object, so we pass the 'train' split of the tokenized_dataset
train_dataset = tokenized_dataset["train"]
# For simplicity, using the same for validation if not explicitly split in data.json
tokenized_data_val = tokenized_dataset["train"]

In [ ]:
from datasets import load_dataset
# Loading data.json instead of processing PDF
dataset = load_dataset('json', data_files='/content/drive/MyDrive/Colab Notebooks/finetune/data.json')

In [ ]:
print(dataset)

In [ ]:
print(dataset["train"].features)

In [ ]:
# Tokenize the dataset using the 'input' column from the 'train' split
tokenized_data = tokenizer(list(dataset["train"]["input"]), truncation=True, padding=True)

# Assuming a validation dataset is also needed for the Trainer,
# for simplicity, we'll use the same tokenized_data for now.
# If you have a separate validation split in your JSON, you would load and tokenize it similarly.
tokenized_data_val = tokenized_data

In [ ]:
tokenized = tokenizer(list(dataset["train"]["question"]), truncation=True, padding=True)

In [ ]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2, # Reduced batch size to mitigate OutOfMemoryError
    num_train_epochs=3,
    learning_rate=5e-5,
    report_to="none" # Disable Weights & Biases logging
)

In [ ]:
from transformers import Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,  # Training dataset, now correctly prepared
)
trainer.train()

In [ ]:
print(dataset["train"].features)

In [ ]:
trainer.save_model("fine-tuned-llm")